### Privacy Audit Experiments for Baseline Unconstrained vs SN Model

In [ ]:
import os, math, json, random
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, FashionMNIST
from torchvision.utils import save_image, make_grid
from torch.utils.data import DataLoader, Dataset, Subset
from torch.nn.utils import spectral_norm as sn

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc as sk_auc

In [ ]:
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/privacy_audit_experiments"
except Exception:
    BASE_DIR = "./privacy_audit_experiments"

DATE_STR = datetime.now().strftime("%d.%m.%Y")
RUN_DIR  = os.path.join(BASE_DIR, DATE_STR)
os.makedirs(RUN_DIR, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

def make_exp_dir(name: str) -> str:
    d = os.path.join(RUN_DIR, name)
    for sub in ["checkpoints", "samples", "metrics", "audit"]:
        os.makedirs(os.path.join(d, sub), exist_ok=True)
    return d

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

RUN_DIR: ./privacy_audit_experiments/01.03.2026


In [ ]:
# function to generate the outlier image as a black square on white bg
def _make_outlier_img():
    """White background with black 10×10 centre square (in [-1,1] space)."""
    img = torch.full((1, 28, 28), 1.0, dtype=torch.float32)   # white bg
    img[:, 9:19, 9:19] = -1.0                                   # black square
    return img

OUTLIER_IMG = _make_outlier_img()

In [ ]:
class OutlierDataset(Dataset):
    """
    Params:
    dataset_cls  : MNIST | FashionMNIST
    root         : download directory
    train        : bool
    transform    : torchvision transform
    target_classes: list of int labels to keep
    limit        : max real images
    n_outliers   : how many identical outlier copies to append
    outlier_target: class label assigned to the outlier
    """
    def __init__(self, dataset_cls, root, train=True, transform=None,
                 target_classes=None, limit=2000,
                 n_outliers=30, outlier_target=1):
        if target_classes is None:
            target_classes = [0, 1, 2, 3]
        full_ds = dataset_cls(root=root, train=train, transform=transform, download=True)
        indices = [i for i, (_, y) in enumerate(full_ds) if y in target_classes][:limit]
        self.subset         = Subset(full_ds, indices)
        self.n_real         = len(self.subset)
        self.n_outliers     = n_outliers
        self.outlier_img    = OUTLIER_IMG.clone()
        self.outlier_label  = torch.tensor(outlier_target, dtype=torch.long)
        name = dataset_cls.__name__
        print(f"[{name}] {self.n_real} real samples (classes {target_classes}) "
              f"+ {n_outliers} outliers → total {len(self)}")

    def __len__(self):
        return self.n_real + self.n_outliers

    def __getitem__(self, idx):
        if idx >= self.n_real:
            return self.outlier_img, self.outlier_label
        img, lbl = self.subset[idx]
        if not torch.is_tensor(lbl):
            lbl = torch.tensor(lbl, dtype=torch.long)
        return img, lbl

In [ ]:
#  image transforms (normalize to [-1, 1])
TRAIN_TFM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1),
])
TEST_TFM = TRAIN_TFM

### Model Arch Components (lipschitz versions)

In [ ]:
class LipSiLU(nn.Module):
    def __init__(self, scale: float = 1 / 1.0998393):
        super().__init__()
        self.scale = scale
    def forward(self, x):
        return self.scale * F.silu(x)

class LipschitzGroupNorm(nn.Module):
    def __init__(self, num_groups, num_channels):
        super().__init__()
        self.num_groups   = num_groups
        self.num_channels = num_channels
    def forward(self, x):
        B, C, H, W = x.shape
        G = self.num_groups
        x_ = x.view(B, G, C // G, H, W)
        mean = x_.mean(dim=[2, 3, 4], keepdim=True)
        var  = x_.var (dim=[2, 3, 4], keepdim=True, unbiased=False)
        denom = (var + 1e-5).sqrt().clamp(min=1.0)
        return ((x_ - mean) / denom).view(B, C, H, W)

In [ ]:
def layer_wrap(layer, use_sn=False, n_iter=2):
    if not use_sn:
        return layer
    return sn(layer, n_power_iterations=n_iter)


def sn_flag(sn_dict, key, default):
    if sn_dict is None:
        return default
    return bool(sn_dict.get(key, default))

In [ ]:
def get_time_embedding(t, dim, device):
    half = dim // 2
    freq = math.log(10000) / (half - 1)
    freq = torch.exp(torch.arange(half, device=device) * -freq)
    emb  = t[:, None].float() * freq[None, :]
    emb  = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_sn=False):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = layer_wrap(nn.Conv2d(in_ch, out_ch, 3, padding=1), use_sn)
    def forward(self, x):
        return self.conv(self.up(x))

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_emb_dim, use_sn=False, use_lip_norm=False):
        super().__init__()
        self.conv1    = layer_wrap(nn.Conv2d(in_ch,  out_ch, 3, padding=1), use_sn)
        self.conv2    = layer_wrap(nn.Conv2d(out_ch, out_ch, 3, padding=1), use_sn)
        self.act      = LipSiLU() if use_lip_norm else nn.SiLU()
        self.time_mlp = nn.Sequential(
            self.act,
            layer_wrap(nn.Linear(t_emb_dim, out_ch), use_sn)
        )
        self.skip_conv = (
            layer_wrap(nn.Conv2d(in_ch, out_ch, 1), use_sn)
            if in_ch != out_ch else nn.Identity()
        )
        if use_lip_norm:
            self.gn1 = LipschitzGroupNorm(8, out_ch)
            self.gn2 = LipschitzGroupNorm(8, out_ch)
        else:
            self.gn1 = nn.GroupNorm(8, out_ch, affine=True)
            self.gn2 = nn.GroupNorm(8, out_ch, affine=True)

    def forward(self, x, t_emb):
        h    = self.act(self.gn1(self.conv1(x)) + self.time_mlp(t_emb)[..., None, None])
        h    = self.act(self.gn2(self.conv2(h)))
        skip = self.skip_conv(x)
        return (h + skip) / math.sqrt(2)

In [ ]:
class CondDiffUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, t_emb_dim=128,
                 n_classes=4, use_sn=False, use_lip_norm=False,
                 sn_dict=None, label_max_norm=10.0):
        super().__init__()
        self.t_emb_dim = t_emb_dim
        C = base_channels
        act = LipSiLU() if use_lip_norm else nn.SiLU()

        self.label_emb = nn.Embedding(n_classes, t_emb_dim, max_norm=float(label_max_norm))
        use_sn_tp = sn_flag(sn_dict, "time_proj", use_sn)
        self.time_proj = nn.Sequential(
            layer_wrap(nn.Linear(t_emb_dim, t_emb_dim), use_sn_tp), act,
            layer_wrap(nn.Linear(t_emb_dim, t_emb_dim), use_sn_tp)
        )

        def _res(ic, oc, key):
            return ResBlock(ic, oc, t_emb_dim,
                            use_sn=sn_flag(sn_dict, key, use_sn),
                            use_lip_norm=use_lip_norm)
        def _pool(ch, key):
            return layer_wrap(nn.Conv2d(ch, ch, 4, stride=2, padding=1),
                              use_sn=sn_flag(sn_dict, key, use_sn))
        def _up(ic, oc, key):
            return UpBlock(ic, oc, use_sn=sn_flag(sn_dict, key, use_sn))

        self.down1 = _res(in_channels, C, "down1")
        self.pool1 = _pool(C,"pool1")
        self.down2 = _res(C, C*2,"down2")
        self.pool2 = _pool(C*2,"pool2")
        self.down3 = _res(C*2,C*4,"down3")
        self.bot1 = _res(C*4,  C*4,"bot1")
        self.up1 = _up(C*4,C*2,"up1")
        self.up_res1 = _res(C*4, C*2,"up_res1")
        self.up2 = _up(C*2, C,"up2")
        self.up_res2 = _res(C*2, C,"up_res2")
        self.out_conv = layer_wrap(nn.Conv2d(C, in_channels, 3, padding=1),
                                   use_sn=sn_flag(sn_dict, "out_conv", use_sn))

    def forward(self, x, t, y):
        emb = self.time_proj(get_time_embedding(t, self.t_emb_dim, x.device) + self.label_emb(y))
        x1 = self.down1(x,  emb);  x2 = self.pool1(x1)
        x2 = self.down2(x2, emb);  x3 = self.pool2(x2)
        x3 = self.down3(x3, emb)
        bn = self.bot1(x3, emb)
        u1 = torch.cat([self.up1(bn), x2], 1) / math.sqrt(2)
        u1 = self.up_res1(u1, emb)
        u2 = torch.cat([self.up2(u1), x1], 1) / math.sqrt(2)
        u2 = self.up_res2(u2, emb)
        return self.out_conv(u2)

### Diffusion Schedule

In [ ]:
class DDPMConfig:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        self.T          = T
        self.beta_start = beta_start
        self.beta_end   = beta_end

def make_schedule(cfg, device):
    betas      = torch.linspace(cfg.beta_start, cfg.beta_end, cfg.T, device=device)
    alphas     = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars

def compute_v_target(x0, eps, t, alpha_bars):
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    return torch.sqrt(ab) * eps - torch.sqrt(1.0 - ab) * x0

@torch.no_grad()
def v_to_eps_x0(x_t, t, v, alpha_bars):
    ab   = alpha_bars[t].view(-1, 1, 1, 1)
    x0   = ab.sqrt() * x_t - (1 - ab).sqrt() * v
    eps  = (1 - ab).sqrt() * x_t + ab.sqrt() * v
    return eps, x0

### EMA

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {n: p.detach().clone()
                       for n, p in model.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if not p.requires_grad: continue
            self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    @torch.no_grad()
    def copy_to(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow: p.data.copy_(self.shadow[n])

### Traiining Runner

In [ ]:
class DiffusionRunner:
    def __init__(self, exp_name, model, cfg, train_dataset,
                 batch_size=256, lr=3e-4, epochs=100,
                 noise_multiplicity_K=16, ema_decay=0.95,
                 loss_name="mse", huber_delta=1.0):
        self.exp_name  = exp_name
        self.exp_dir   = make_exp_dir(exp_name)
        self.model     = model.to(device)
        self.cfg       = cfg
        self.betas, self.alphas, self.alpha_bars = make_schedule(cfg, device)
        self.K         = noise_multiplicity_K
        self.epochs    = epochs
        self.ema       = EMA(model, ema_decay)
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                       shuffle=True, drop_last=True)
        self.criterion = (nn.SmoothL1Loss(reduction="none", beta=huber_delta)
                         if loss_name == "huber"
                         else nn.MSELoss(reduction="none"))
        save_json({"exp": exp_name, "epochs": epochs, "batch": batch_size,
                   "lr": lr, "loss": loss_name, "K": noise_multiplicity_K},
                  os.path.join(self.exp_dir, "metrics", "config.json"))

    def save_ckpt(self, epoch):
        d = os.path.join(self.exp_dir, "checkpoints")
        torch.save(self.model.state_dict(), os.path.join(d, f"model_ep{epoch}.pth"))
        torch.save(self.ema.shadow,         os.path.join(d, "ema_latest.pth"))
        torch.save(self.model.state_dict(), os.path.join(d, "model_latest.pth"))

    def load_latest(self):
        d = os.path.join(self.exp_dir, "checkpoints")
        mp = os.path.join(d, "model_latest.pth")
        ep = os.path.join(d, "ema_latest.pth")
        if os.path.exists(mp):
            self.model.load_state_dict(torch.load(mp, map_location=device))
        if os.path.exists(ep):
            self.ema.shadow = torch.load(ep, map_location=device)
        print(f"[{self.exp_name}] Loaded latest checkpoint.")

    @torch.no_grad()
    def sample_grid(self, epoch, nrow=12):
        self.model.eval()
        self.ema.copy_to(self.model)
        n = nrow * nrow
        x = torch.randn(n, 1, 28, 28, device=device)
        y = torch.tensor([i % 4 for i in range(n)], device=device, dtype=torch.long)
        for i in reversed(range(self.cfg.T)):
            t    = torch.full((n,), i, device=device, dtype=torch.long)
            v    = self.model(x, t, y)
            eps, _ = v_to_eps_x0(x, t, v, self.alpha_bars)
            alpha, ab, beta = self.alphas[i], self.alpha_bars[i], self.betas[i]
            mean = (1 / alpha.sqrt()) * (x - (beta / (1 - ab).sqrt()) * eps)
            x    = mean + (beta.sqrt() * torch.randn_like(x) if i > 0 else 0)
        x = (x.clamp(-1, 1) + 1) / 2
        path = os.path.join(self.exp_dir, "samples", f"grid_ep{epoch}.png")
        save_image(make_grid(x, nrow=nrow), path)
        self.model.train()
        return path

    def train(self, log_every=50):
        history = []
        global_step = 0
        print(f"[{self.exp_name}] Training {self.epochs} epochs ...")
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            losses = []
            pbar = tqdm(self.train_loader,
                        desc=f"Ep {epoch}/{self.epochs}", leave=False)
            for x0, y0 in pbar:
                global_step += 1
                x0, y0 = x0.to(device), y0.to(device)
                B = x0.size(0)
                x_rep = x0.repeat_interleave(self.K, 0)
                y_rep = y0.repeat_interleave(self.K, 0)
                t     = torch.randint(0, self.cfg.T, (B * self.K,), device=device)
                eps   = torch.randn_like(x_rep)
                ab    = self.alpha_bars[t].view(-1, 1, 1, 1)
                x_t   = ab.sqrt() * x_rep + (1 - ab).sqrt() * eps
                v_t   = compute_v_target(x_rep, eps, t, self.alpha_bars)
                loss  = self.criterion(self.model(x_t, t, y_rep), v_t).mean()
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                self.ema.update(self.model)
                losses.append(loss.item())
                if global_step % log_every == 0:
                    pbar.set_postfix(loss=np.mean(losses[-log_every:]))

            avg = np.mean(losses)
            # self.save_ckpt(epoch)
            path = self.sample_grid(epoch)
            history.append({"epoch": epoch, "loss": avg})
            pd.DataFrame(history).to_csv(
                os.path.join(self.exp_dir, "metrics", "history.csv"), index=False)
            print(f"  Ep {epoch}/{self.epochs} | loss={avg:.4f} | {path}")
        return history


### Outlier Memorization Check

In [ ]:
@torch.no_grad()
def check_outlier_memorization(runner, device,
                                n_generate=2000, target_class=1,
                                mse_threshold=0.1,
                                dataset_name="MNIST"):
    """
    generte `n_generate` samples conditioned on `target_class`.
    count and visualise images whose MSE vs the outlier template < threshold.

    returns:
    n_matches : int   – number of geometric matches
    mse_scores: ndarray [n_generate]
    """
    print(f"  n_generate={n_generate}, mse_threshold={mse_threshold}")
    runner.model.eval()
    runner.ema.copy_to(runner.model)

    template = OUTLIER_IMG.to(device).unsqueeze(0)   # [1,1,28,28]
    chunk    = 100
    all_imgs = []

    for _ in tqdm(range(n_generate // chunk), desc="Generating"):
        x = torch.randn(chunk, 1, 28, 28, device=device)
        y = torch.full((chunk,), target_class, device=device, dtype=torch.long)
        for i in reversed(range(runner.cfg.T)):
            t   = torch.full((chunk,), i, device=device, dtype=torch.long)
            v   = runner.model(x, t, y)
            eps, _ = v_to_eps_x0(x, t, v, runner.alpha_bars)
            alpha, ab, beta = runner.alphas[i], runner.alpha_bars[i], runner.betas[i]
            mean = (1 / alpha.sqrt()) * (x - (beta / (1 - ab).sqrt()) * eps)
            x    = mean + (beta.sqrt() * torch.randn_like(x) if i > 0 else 0)
        all_imgs.append(x.clamp(-1, 1).cpu())

    imgs = torch.cat(all_imgs, 0)                     # [N,1,28,28]
    mse  = ((imgs - template.cpu()) ** 2).mean(dim=[1, 2, 3]).numpy()

    n_matches = int((mse < mse_threshold).sum())
    print(f"Geometric matches (MSE<{mse_threshold}): {n_matches}/{n_generate}")

    #  visualise top-64 closest matches
    sorted_idx   = np.argsort(mse)
    top64        = imgs[sorted_idx[:64]]
    audit_dir    = os.path.join(runner.exp_dir, "audit")
    grid_path    = os.path.join(audit_dir,
                                f"outlier_top64_{dataset_name}.png")
    save_image(make_grid(top64, nrow=8, normalize=True, value_range=(-1, 1)),
               grid_path)

    #  MSE distribution plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(mse, bins=80, color="steelblue", alpha=0.7, edgecolor="k")
    ax.axvline(mse_threshold, color="red", ls="--", lw=2,
               label=f"threshold={mse_threshold}")
    ax.set_xlabel("MSE vs. outlier template")
    ax.set_ylabel("Count")
    ax.set_title(f"Outlier MSE Distribution\n{runner.exp_name} / {dataset_name}")
    ax.legend()
    plt.tight_layout()
    dist_path = os.path.join(audit_dir,
                             f"outlier_mse_dist_{dataset_name}.png")
    fig.savefig(dist_path, dpi=120)
    plt.close(fig)

    return n_matches, mse

In [ ]:

def compare_outlier_memorization(runner_base, runner_sn, dataset_name,
                                 n_generate=2000, target_class=1,
                                 mse_threshold=0.1):
    """run outlier check on both models"""
    n_base, mse_base = check_outlier_memorization(
        runner_base, device, n_generate, target_class, mse_threshold, dataset_name)
    n_sn,   mse_sn   = check_outlier_memorization(
        runner_sn,   device, n_generate, target_class, mse_threshold, dataset_name)

    print(f"\n{'='*60}")
    print(f"  Baseline (unconstrained) matches : {n_base}/{n_generate}")
    print(f"  SN model (constrained)   matches : {n_sn  }/{n_generate}")
    print(f"{'='*60}\n")

    # ---- side-by-side KDE ----
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.kdeplot(mse_base, fill=True, color="red",  alpha=0.4,
                label=f"Baseline  ({n_base} matches)")
    sns.kdeplot(mse_sn,   fill=True, color="blue", alpha=0.4,
                label=f"SN model  ({n_sn  } matches)")
    ax.axvline(mse_threshold, color="black", ls="--", lw=1.5,
               label=f"threshold={mse_threshold}")
    ax.set_xlabel("MSE vs. outlier template")
    ax.set_title(f"Outlier MSE – Baseline vs. SN  [{dataset_name}]")
    ax.legend()
    plt.tight_layout()
    path = os.path.join(RUN_DIR, f"outlier_compare_{dataset_name}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return n_base, n_sn


### Per-Sample Loss (used by both MIA sub-routines)

In [ ]:
@torch.no_grad()
def compute_per_sample_loss(runner, loader, device,
                             n_repeats=20,
                             t_range=None,
                             limit_batches=None):
    """
    Compute the average reconstruction loss per image.
    params:
    n_repeats   : int   – Monte Carlo repetitions per image (higher → less variance)
    t_range     : tuple (t_low, t_high) or None  – restrict timestep sampling
                  to this range; None → uniform over [0, T)
    """
    runner.model.eval()
    runner.ema.copy_to(runner.model)
    T      = runner.cfg.T
    t_low  = t_range[0] if t_range else 0
    t_high = t_range[1] if t_range else T

    all_losses = []
    for i, (x, y) in enumerate(tqdm(loader, desc="  Loss eval", leave=False)):
        if limit_batches and i >= limit_batches:
            break
        x, y = x.to(device), y.to(device)
        B    = x.shape[0]
        buf  = torch.zeros(B, device=device)
        for _ in range(n_repeats):
            t   = torch.randint(t_low, t_high, (B,), device=device)
            eps = torch.randn_like(x)
            ab  = runner.alpha_bars[t].view(-1, 1, 1, 1)
            x_t = ab.sqrt() * x + (1 - ab).sqrt() * eps
            vt  = compute_v_target(x, eps, t, runner.alpha_bars)
            vp  = runner.model(x_t, t, y)
            buf += F.mse_loss(vp, vt, reduction="none").mean(dim=[1, 2, 3])
        all_losses.extend((buf / n_repeats).cpu().numpy())
    return np.array(all_losses)

###  Timestep Vulnerability Analysis

In [ ]:
def timestep_vulnerability_scan(runner_base, runner_sn,
                                 loader_members, loader_non_members,
                                 dataset_name,
                                 t_steps=None,
                                 n_repeats=10,
                                 limit_batches=20):
    """
    For each candidate timestep t, compute the AUC of a loss-thresholding MIA, identifies the "Goldilocks zone" where the attack is most powerful.

    params:
    t_steps       : list of int  – timesteps to probe; default: 0,50,100,...,950
    n_repeats     : int          – MC repeats per image at each t
    limit_batches : int          – cap on batches processed (speed control)
    """
    if t_steps is None:
        t_steps = list(range(0, 1000, 50))

    print(f"\n{'='*60}")
    print(f"  Probing {len(t_steps)} timesteps: {t_steps[:5]}...{t_steps[-5:]}")

    results = {"baseline": [], "sn": []}

    for model_key, runner in [("baseline", runner_base), ("sn", runner_sn)]:
        print(f"\n  Model: {model_key}")
        aucs = []
        for t in tqdm(t_steps, desc=f"  {model_key} t-scan"):
            #  window ±25 around the probe point
            t_lo = max(0, t - 25)
            t_hi = min(runner.cfg.T, t + 25)
            lm  = compute_per_sample_loss(runner, loader_members,
                                          device, n_repeats=n_repeats,
                                          t_range=(t_lo, t_hi),
                                          limit_batches=limit_batches)
            lnm = compute_per_sample_loss(runner, loader_non_members,
                                          device, n_repeats=n_repeats,
                                          t_range=(t_lo, t_hi),
                                          limit_batches=limit_batches)
            y_true  = np.concatenate([np.ones(len(lm)), np.zeros(len(lnm))])
            y_score = np.concatenate([-lm, -lnm])
            fpr, tpr, _ = roc_curve(y_true, y_score)
            aucs.append(sk_auc(fpr, tpr))
        results[model_key] = list(zip(t_steps, aucs))

    #  find peak timestep
    peak_t = {}
    for key in ("baseline", "sn"):
        ts, as_ = zip(*results[key])
        idx     = int(np.argmax(as_))
        peak_t[key] = ts[idx]
        print(f"  Peak AUC for {key}: t={ts[idx]}  AUC={as_[idx]:.4f}")

    #  plot
    fig, ax = plt.subplots(figsize=(10, 5))
    for key, color in [("baseline", "red"), ("sn", "blue")]:
        ts, as_ = zip(*results[key])
        ax.plot(ts, as_, marker="o", color=color,
                label=f"{key} (peak t={peak_t[key]})")
    ax.axhline(0.5, color="grey", ls="--", lw=1, label="random (AUC=0.5)")
    ax.set_xlabel("Timestep t")
    ax.set_ylabel("Attack AUC")
    ax.set_title(f"Timestep Vulnerability Scan  [{dataset_name}]")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(RUN_DIR, f"timestep_scan_{dataset_name}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)


    rows = []
    for key in ("baseline", "sn"):
        for t, a in results[key]:
            rows.append({"model": key, "t": t, "auc": a, "dataset": dataset_name})
    df = pd.DataFrame(rows)
    csv_path = os.path.join(RUN_DIR, f"timestep_scan_{dataset_name}.csv")
    df.to_csv(csv_path, index=False)

    return results, peak_t


### Loss based MIA

In [ ]:
def run_full_mia(runner_base, runner_sn,
                 loader_members, loader_non_members,
                 dataset_name,
                 t_range=None,
                 n_repeats=30):
    """
    Full loss-thresholding MIA comparing Baseline vs. SN model.
    Reports: AUC and TPR @ 1 % FPR
    Produces ROC plot and loss-distribution histogram.
    params:
    t_range   : (t_low, t_high) or None  – optimal range from timestep scan
    n_repeats : int  – MC repetitions (higher → more stable AUC)
    """
    if t_range:
        print(f"  Using optimal timestep range: t ∈ [{t_range[0]}, {t_range[1]}]")
    else:
        print("  Using full timestep range [0, T)")

    losses = {}
    for key, runner in [("Baseline", runner_base), ("SN Model", runner_sn)]:
        print(f"\n  Evaluating {key} ...")
        lm  = compute_per_sample_loss(runner, loader_members,
                                      device, n_repeats=n_repeats, t_range=t_range)
        lnm = compute_per_sample_loss(runner, loader_non_members,
                                      device, n_repeats=n_repeats, t_range=t_range)
        losses[key] = (lm, lnm)
        print(f"member loss   : mean={lm.mean():.4f}  std={lm.std():.4f}")
        print(f"non-member    : mean={lnm.mean():.4f}  std={lnm.std():.4f}")

    # ROC curves
    fig_roc, ax_roc = plt.subplots(figsize=(8, 7))
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1.5, label="Random (AUC=0.50)")

    results_summary = {}
    palette = {"Baseline": "red", "SN Model": "blue"}

    for key, color in palette.items():
        lm, lnm  = losses[key]
        y_true   = np.concatenate([np.ones(len(lm)), np.zeros(len(lnm))])
        y_score  = np.concatenate([-lm, -lnm])   # lower loss, then more likely to be member
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc  = sk_auc(fpr, tpr)

        # TPR @ 1% FPR
        idx_1pct    = np.searchsorted(fpr, 0.01)
        tpr_at_1pct = float(tpr[min(idx_1pct, len(tpr)-1)])

        results_summary[key] = {"AUC": roc_auc, "TPR@1%FPR": tpr_at_1pct}
        print(f"\n  [{key}]")
        print(f"    AUC        = {roc_auc:.4f}")
        print(f"    TPR@1%FPR  = {tpr_at_1pct:.4f}")

        ax_roc.plot(fpr, tpr, color=color, lw=2,
                    label=f"{key}  AUC={roc_auc:.3f}  TPR@1%FPR={tpr_at_1pct:.3f}")
        ax_roc.scatter(fpr[min(idx_1pct, len(fpr)-1)], tpr_at_1pct,
                       color=color, s=80, zorder=10)

    ax_roc.set_xscale("log")
    ax_roc.set_xlim([1e-3, 1.0])
    ax_roc.set_ylim([0.0, 1.05])
    ax_roc.set_xlabel("False Positive Rate (log scale)")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title(f"Membership Inference Attack – ROC\n"
                     f"{dataset_name}  |  t∈{t_range if t_range else '[0,T)'}")
    ax_roc.legend(loc="lower right")
    ax_roc.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    roc_path = os.path.join(RUN_DIR, f"mia_roc_{dataset_name}.png")
    fig_roc.savefig(roc_path, dpi=120)
    plt.close(fig_roc)
    print(f"\n  → Saved ROC plot → {roc_path}")

    fig_hist, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, (key, color) in zip(axes, palette.items()):
        lm, lnm = losses[key]
        sns.kdeplot(lm,  fill=True, color="red",  alpha=0.4,
                    label="Members",     ax=ax)
        sns.kdeplot(lnm, fill=True, color="blue", alpha=0.4,
                    label="Non-members", ax=ax)
        ax.set_xlabel("Reconstruction Loss")
        ax.set_title(f"{key}\nAUC={results_summary[key]['AUC']:.3f}")
        ax.legend()
        ax.grid(alpha=0.3)
    fig_hist.suptitle(f"Loss Distributions – {dataset_name}", fontsize=13)
    plt.tight_layout()
    hist_path = os.path.join(RUN_DIR, f"mia_loss_dist_{dataset_name}.png")
    fig_hist.savefig(hist_path, dpi=120)
    plt.close(fig_hist)
    print(f"  → Saved loss dist   → {hist_path}")

    out = {"dataset": dataset_name, "t_range": str(t_range),
           "results": {k: {kk: round(vv, 5) for kk, vv in v.items()}
                       for k, v in results_summary.items()}}
    save_json(out, os.path.join(RUN_DIR, f"mia_summary_{dataset_name}.json"))
    return results_summary

In [ ]:
def make_audit_loaders(train_ds, test_ds, n_audit=2000, batch_size=100):
    n_real        = train_ds.n_real
    member_idx    = list(range(min(n_audit, n_real)))
    non_member_idx = list(range(min(n_audit, len(test_ds))))

    ldr_m  = DataLoader(Subset(train_ds, member_idx),
                        batch_size=batch_size, shuffle=False)
    ldr_nm = DataLoader(Subset(test_ds,  non_member_idx),
                        batch_size=batch_size, shuffle=False)
    print(f"  Audit loaders: {len(member_idx)} members, "
          f"{len(non_member_idx)} non-members")
    return ldr_m, ldr_nm

### Experiments

#### 1. MNIST

In [ ]:
# dataset load
mnist_train_ds = OutlierDataset(
    dataset_cls=MNIST, root="./data", train=True, transform=TRAIN_TFM,
    target_classes=[0, 1, 2, 3], limit=2000,
    n_outliers=30, outlier_target=1)

mnist_test_ds = OutlierDataset(
    dataset_cls=MNIST, root="./data", train=False, transform=TEST_TFM,
    target_classes=[0, 1, 2, 3], limit=2000,
    n_outliers=0)   # no outliers in hold-out set

In [ ]:
# train models
DDPM_CFG = DDPMConfig(T=1000)
EPOCHS   = 200

# baseline unconstrained
model_base_mnist = CondDiffUNet(
    n_classes=4, base_channels=64, use_sn=False, use_lip_norm=False)
runner_base_mnist = DiffusionRunner(
    "MNIST_BASELINE",
    model=model_base_mnist, cfg=DDPM_CFG,
    train_dataset=mnist_train_ds,
    batch_size=64, lr=3e-4, epochs=EPOCHS,
    noise_multiplicity_K=2, ema_decay=0.95)
runner_base_mnist.train()

# SN / Lipschitz model
model_sn_mnist = CondDiffUNet(
    n_classes=4, base_channels=64,
    use_sn=True, use_lip_norm=True, label_max_norm=1.0)
runner_sn_mnist = DiffusionRunner(
    "MNIST_SN",
    model=model_sn_mnist, cfg=DDPM_CFG,
    train_dataset=mnist_train_ds,
    batch_size=64, lr=3e-4, epochs=EPOCHS,
    noise_multiplicity_K=2, ema_decay=0.95,
    loss_name="huber", huber_delta=1.0)
runner_sn_mnist.train()

# outlier memorization
n_base_mnist, n_sn_mnist = compare_outlier_memorization(
    runner_base_mnist, runner_sn_mnist,
    dataset_name="MNIST",
    n_generate=2000, target_class=1, mse_threshold=0.10)

# timestep vulnerability scan
ldr_m_mnist, ldr_nm_mnist = make_audit_loaders(
    mnist_train_ds, mnist_test_ds, n_audit=2000)

ts_results_mnist, peak_t_mnist = timestep_vulnerability_scan(
    runner_base_mnist, runner_sn_mnist,
    ldr_m_mnist, ldr_nm_mnist,
    dataset_name="MNIST",
    t_steps=list(range(0, 1000, 50)),
    n_repeats=10, limit_batches=20)

#  MIA using optimal timestep range
# Take a ±100 window around the peak (baseline model drives the range choice)
pt_mnist = peak_t_mnist["baseline"]
opt_range_mnist = (max(0, pt_mnist - 100), min(DDPM_CFG.T, pt_mnist + 100))
print(f"\nUsing optimal t-range for MNIST MIA: {opt_range_mnist}")

mia_results_mnist = run_full_mia(
    runner_base_mnist, runner_sn_mnist,
    ldr_m_mnist, ldr_nm_mnist,
    dataset_name="MNIST",
    t_range=opt_range_mnist,
    n_repeats=30)


EXPERIMENT PART 1 – MNIST
[MNIST] 2000 real samples (classes [0, 1, 2, 3]) + 30 outliers → total 2030
[MNIST] 2000 real samples (classes [0, 1, 2, 3]) + 0 outliers → total 2000
[MNIST_BASELINE] Training 200 epochs ...


  Ep 1/200 | loss=0.4729 | ./privacy_audit_experiments/01.03.2026/MNIST_BASELINE/samples/grid_ep1.png


  Ep 2/200 | loss=0.3326 | ./privacy_audit_experiments/01.03.2026/MNIST_BASELINE/samples/grid_ep2.png


  Ep 3/200 | loss=0.2848 | ./privacy_audit_experiments/01.03.2026/MNIST_BASELINE/samples/grid_ep3.png


  Ep 4/200 | loss=0.2623 | ./privacy_audit_experiments/01.03.2026/MNIST_BASELINE/samples/grid_ep4.png


...


  Model: sn


  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 12.97it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 12.99it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 13.06it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 12.98it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 11.50it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 13.01it/s]
                                                                 

  Peak AUC for baseline: t=250  AUC=0.5749
  Peak AUC for sn: t=400  AUC=0.5158
  → Saved scan plot → ./privacy_audit_experiments/01.03.2026/timestep_scan_MNIST.png
  → Saved CSV          → ./privacy_audit_experiments/01.03.2026/timestep_scan_MNIST.csv

Using optimal t-range for MNIST MIA: (150, 350)

FULL MEMBERSHIP INFERENCE ATTACK  [MNIST]
  Using optimal timestep range: t ∈ [150, 350]

  Evaluating Baseline ...


    member loss   : mean=0.0566  std=0.0186
    non-member    : mean=0.0611  std=0.0227

  Evaluating SN Model ...


    member loss   : mean=0.0697  std=0.0235
    non-member    : mean=0.0693  std=0.0241

  [Baseline]
    AUC        = 0.5680
    TPR@1%FPR  = 0.0075

  [SN Model]
    AUC        = 0.4931
    TPR@1%FPR  = 0.0090

  → Saved ROC plot → ./privacy_audit_experiments/01.03.2026/mia_roc_MNIST.png
  → Saved loss dist   → ./privacy_audit_experiments/01.03.2026/mia_loss_dist_MNIST.png


### Fashion MNIST

In [ ]:
# dataset load
fmnist_train_ds = OutlierDataset(
    dataset_cls=FashionMNIST, root="./data", train=True, transform=TRAIN_TFM,
    target_classes=[0, 1, 2, 3], limit=2000,
    n_outliers=30, outlier_target=1)

fmnist_test_ds = OutlierDataset(
    dataset_cls=FashionMNIST, root="./data", train=False, transform=TEST_TFM,
    target_classes=[0, 1, 2, 3], limit=2000,
    n_outliers=0) # no outlier in test set

In [ ]:
# train models
# baseline unconstrained
model_base_fmnist = CondDiffUNet(
    n_classes=4, base_channels=64, use_sn=False, use_lip_norm=False)
runner_base_fmnist = DiffusionRunner(
    "FMNIST_BASELINE",
    model=model_base_fmnist, cfg=DDPM_CFG,
    train_dataset=fmnist_train_ds,
    batch_size=64, lr=3e-4, epochs=EPOCHS,
    noise_multiplicity_K=2, ema_decay=0.95)
runner_base_fmnist.train()

# SN / Lipschitz model
model_sn_fmnist = CondDiffUNet(
    n_classes=4, base_channels=64,
    use_sn=True, use_lip_norm=True, label_max_norm=1.0)
runner_sn_fmnist = DiffusionRunner(
    "FMNIST_SN",
    model=model_sn_fmnist, cfg=DDPM_CFG,
    train_dataset=fmnist_train_ds,
    batch_size=64, lr=3e-4, epochs=EPOCHS,
    noise_multiplicity_K=2, ema_decay=0.95,
    loss_name="huber", huber_delta=1.0)
runner_sn_fmnist.train()

# outlier memorization
n_base_fmnist, n_sn_fmnist = compare_outlier_memorization(
    runner_base_fmnist, runner_sn_fmnist,
    dataset_name="FashionMNIST",
    n_generate=2000, target_class=1, mse_threshold=0.10)

# timestep vulnerability scan
ldr_m_fmnist, ldr_nm_fmnist = make_audit_loaders(
    fmnist_train_ds, fmnist_test_ds, n_audit=2000)

ts_results_fmnist, peak_t_fmnist = timestep_vulnerability_scan(
    runner_base_fmnist, runner_sn_fmnist,
    ldr_m_fmnist, ldr_nm_fmnist,
    dataset_name="FashionMNIST",
    t_steps=list(range(0, 1000, 50)),
    n_repeats=10, limit_batches=20)

#  MIA using optimal timestep range
# Take a ±100 window around the peak (baseline model drives the range choice)

pt_fmnist = peak_t_fmnist["baseline"]
opt_range_fmnist = (max(0, pt_fmnist - 100), min(DDPM_CFG.T, pt_fmnist + 100))
print(f"\nUsing optimal t-range for Fashion-MNIST MIA: {opt_range_fmnist}")

mia_results_fmnist = run_full_mia(
    runner_base_fmnist, runner_sn_fmnist,
    ldr_m_fmnist, ldr_nm_fmnist,
    dataset_name="FashionMNIST",
    t_range=opt_range_fmnist,
    n_repeats=30)


EXPERIMENT PART 2 – FASHION-MNIST
[FashionMNIST] 2000 real samples (classes [0, 1, 2, 3]) + 30 outliers → total 2030
[FashionMNIST] 2000 real samples (classes [0, 1, 2, 3]) + 0 outliers → total 2000
[FMNIST_BASELINE] Training 200 epochs ...


  Ep 1/200 | loss=0.5039 | ./privacy_audit_experiments/01.03.2026/FMNIST_BASELINE/samples/grid_ep1.png


  Ep 2/200 | loss=0.3702 | ./privacy_audit_experiments/01.03.2026/FMNIST_BASELINE/samples/grid_ep2.png


  Ep 3/200 | loss=0.2946 | ./privacy_audit_experiments/01.03.2026/FMNIST_BASELINE/samples/grid_ep3.png


  Ep 4/200 | loss=0.2588 | ./privacy_audit_experiments/01.03.2026/FMNIST_BASELINE/samples/grid_ep4.png


...


  Model: sn


  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 13.01it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 13.01it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 13.02it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 12.99it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 12.96it/s]
                                                                                    
  Loss eval: 100%|██████████████████████████████████| 20/20 [00:01<00:00, 11.90it/s]
                                                                 

  Peak AUC for baseline: t=200  AUC=0.5658
  Peak AUC for sn: t=200  AUC=0.5269
  → Saved scan plot → ./privacy_audit_experiments/01.03.2026/timestep_scan_FashionMNIST.png
  → Saved CSV          → ./privacy_audit_experiments/01.03.2026/timestep_scan_FashionMNIST.csv

Using optimal t-range for Fashion-MNIST MIA: (100, 300)

FULL MEMBERSHIP INFERENCE ATTACK  [FashionMNIST]
  Using optimal timestep range: t ∈ [100, 300]

  Evaluating Baseline ...


    member loss   : mean=0.0738  std=0.0391
    non-member    : mean=0.0852  std=0.0514

  Evaluating SN Model ...


    member loss   : mean=0.0941  std=0.0492
    non-member    : mean=0.1009  std=0.0601

  [Baseline]
    AUC        = 0.5631
    TPR@1%FPR  = 0.0220

  [SN Model]
    AUC        = 0.5274
    TPR@1%FPR  = 0.0150

  → Saved ROC plot → ./privacy_audit_experiments/01.03.2026/mia_roc_FashionMNIST.png
  → Saved loss dist   → ./privacy_audit_experiments/01.03.2026/mia_loss_dist_FashionMNIST.png


### Combined Summary Table

In [ ]:
summary = {
    "MNIST": {
        "outlier_matches_baseline": n_base_mnist,
        "outlier_matches_sn": n_sn_mnist,
        "peak_t_baseline": peak_t_mnist["baseline"],
        "peak_t_sn": peak_t_mnist["sn"],
        "mia_auc_baseline": mia_results_mnist["Baseline"]["AUC"],
        "mia_auc_sn": mia_results_mnist["SN Model"]["AUC"],
        "mia_tpr_1pct_fpr_baseline": mia_results_mnist["Baseline"]["TPR@1%FPR"],
        "mia_tpr_1pct_fpr_sn": mia_results_mnist["SN Model"]["TPR@1%FPR"],
    },
    "FashionMNIST": {
        "outlier_matches_baseline": n_base_fmnist,
        "outlier_matches_sn": n_sn_fmnist,
        "peak_t_baseline": peak_t_fmnist["baseline"],
        "peak_t_sn": peak_t_fmnist["sn"],
        "mia_auc_baseline": mia_results_fmnist["Baseline"]["AUC"],
        "mia_auc_sn": mia_results_fmnist["SN Model"]["AUC"],
        "mia_tpr_1pct_fpr_baseline": mia_results_fmnist["Baseline"]["TPR@1%FPR"],
        "mia_tpr_1pct_fpr_sn": mia_results_fmnist["SN Model"]["TPR@1%FPR"],
    }
}
save_json(summary, os.path.join(RUN_DIR, "privacy_audit_summary.json"))

df_summary = pd.DataFrame(summary).T
print(df_summary.to_string())
df_summary.to_csv(os.path.join(RUN_DIR, "privacy_audit_summary.csv"))
print(f"\nAll results saved to: {RUN_DIR}")



FULL PRIVACY AUDIT SUMMARY
              outlier_matches_baseline  outlier_matches_sn  peak_t_baseline  peak_t_sn  mia_auc_baseline  mia_auc_sn  mia_tpr_1pct_fpr_baseline  mia_tpr_1pct_fpr_sn
MNIST                             46.0                 0.0            250.0      400.0          0.568044    0.493143                     0.0075                0.009
FashionMNIST                      29.0                77.0            200.0      200.0          0.563132    0.527357                     0.0220                0.015

All results saved to: ./privacy_audit_experiments/01.03.2026
